# 01 — Tokenisasi: Dari Teks Menjadi Angka

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/06-transformers-nlp/01_tokenisasi.ipynb)

Model hanya bisa memproses angka. **Tokenizer** memotong teks menjadi potongan (*token*) dan memetakan setiap token ke sebuah ID. Di modul 05, ViT memotong gambar menjadi patch; tokenizer melakukan hal yang sama untuk teks.

Pilihan tokenizer ternyata sangat berpengaruh: menentukan panjang input, biaya API LLM (yang dihitung per token), dan seberapa baik model memahami bahasa tertentu, **termasuk bahasa Indonesia**.

Isi notebook:
1. Tiga tingkat tokenisasi: karakter, kata, dan **subword**.
2. **Byte-Pair Encoding (BPE)** dari nol.
3. Melatih tokenizer BPE dan **WordPiece** sendiri dengan library `tokenizers`.
4. Membandingkan tokenizer pretrained (GPT-2, BERT, IndoBERT, XLM-R) pada teks bahasa Indonesia.
5. Anatomi output tokenizer Hugging Face: `input_ids`, `attention_mask`, padding, dan truncation.

**Dataset: NusaX-Senti (bahasa Indonesia)**, kalimat ulasan/komentar berlabel `negative`, `neutral`, `positive` (500 train, 100 validasi, 400 test). Sumber: [IndoNLP/nusax](https://github.com/IndoNLP/nusax), lisensi data **CC-BY-SA**. Data diunduh langsung dari GitHub dan tidak disimpan di repo ini.

Notebook ini tidak butuh GPU.

## 0. Setup

`transformers` dan `tokenizers` (Apache-2.0) sudah terpasang di Colab; sel ini memastikan versinya cukup baru.

In [ ]:
%pip install -q "transformers>=4.46" tokenizers pandas

Import library dan unduh teks NusaX (split train) sebagai korpus.

In [ ]:
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

NUSAX_URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment/indonesian"
train_df = pd.read_csv(f"{NUSAX_URL}/train.csv")
korpus = train_df["text"].tolist()
print(f"{len(korpus)} kalimat | contoh: {korpus[2]!r}")

## 1. Karakter, Kata, atau Subword?

| Tingkat | Contoh `"makanannya enak"` | Kelebihan | Kekurangan |
|---|---|---|---|
| **Karakter** | `m a k a n a n n y a ␣ e n a k` | vocab kecil (~100), tidak ada kata asing | deretan sangat panjang, makna per token lemah |
| **Kata** | `makanannya`, `enak` | deretan pendek, bermakna | vocab raksasa; kata baru/typo menjadi `[UNK]` (*out-of-vocabulary*) |
| **Subword** | `makan`, `##annya`, `enak` | kompromi: kata umum utuh, kata jarang dipecah jadi bagian yang dikenal | perlu algoritma untuk memilih potongan |

Bahasa Indonesia kaya imbuhan (`makan` → `makanan`, `dimakan`, `memakannya`, `termakan`), sehingga tokenisasi per kata menghasilkan vocab yang sangat besar. Subword menangani ini dengan elegan.

Mari lihat masalah OOV secara nyata: berapa banyak kata di data **test** yang tidak pernah muncul di data train?

In [ ]:
def kata(teks):
    return re.findall(r"\w+|[^\w\s]", teks.lower())

vocab_kata = Counter(k for t in korpus for k in kata(t))
test_df = pd.read_csv(f"{NUSAX_URL}/test.csv")
kata_test = [k for t in test_df["text"] for k in kata(t)]
oov = [k for k in kata_test if k not in vocab_kata]

print(f"Vocab kata dari 500 kalimat train : {len(vocab_kata):,}")
print(f"Kata di test yang tidak dikenal    : {len(oov) / len(kata_test):.1%}")
print("Contoh kata OOV:", sorted(set(oov))[:15])
print(f"Vocab karakter                     : {len(set(''.join(korpus).lower()))}")

## 2. Byte-Pair Encoding (BPE) dari Nol

BPE (dipakai GPT, LLaMA, dan banyak LLM) membangun vocab secara bertahap:

1. Mulai dari vocab **karakter**. Setiap kata dipecah menjadi karakter, ditambah penanda akhir kata `</w>`.
2. Hitung semua **pasangan simbol berurutan** di korpus.
3. **Gabungkan** pasangan yang paling sering muncul menjadi satu simbol baru, dan catat sebagai *aturan merge*.
4. Ulangi sampai jumlah merge (ukuran vocab) yang diinginkan tercapai.

Kita pakai korpus mainan berisi kata-kata berimbuhan dengan frekuensinya.

In [ ]:
korpus_mainan = {"makan": 10, "makanan": 6, "dimakan": 4, "minum": 8, "minuman": 5, "diminum": 3, "makanannya": 2}

def pecah(kata):
    return tuple(kata) + ("</w>",)

def hitung_pasangan(vocab):
    pasangan = Counter()
    for simbol, frek in vocab.items():
        for a, b in zip(simbol, simbol[1:]):
            pasangan[(a, b)] += frek
    return pasangan

def gabung(vocab, pasangan):
    a, b = pasangan
    baru = {}
    for simbol, frek in vocab.items():
        hasil, i = [], 0
        while i < len(simbol):
            if i < len(simbol) - 1 and (simbol[i], simbol[i + 1]) == (a, b):
                hasil.append(a + b); i += 2
            else:
                hasil.append(simbol[i]); i += 1
        baru[tuple(hasil)] = frek
    return baru

vocab = {pecah(k): f for k, f in korpus_mainan.items()}
merges = []
for langkah in range(12):
    pasangan = hitung_pasangan(vocab)
    terbaik, frek = pasangan.most_common(1)[0]
    vocab = gabung(vocab, terbaik)
    merges.append(terbaik)
    print(f"merge {langkah + 1:2d}: {terbaik[0]!r:>8} + {terbaik[1]!r:<8} (frek {frek:2d})")

print("\nHasil segmentasi korpus:")
for simbol in vocab:
    print("  ", " ".join(simbol))

Setelah dilatih, **aturan merge diterapkan berurutan** ke kata apa pun, termasuk kata yang tidak ada di korpus. Kata baru seperti `"meminum"` tetap bisa dipecah menjadi potongan yang dikenal, tanpa `[UNK]`.

In [ ]:
def encode_bpe(kata, merges):
    simbol = pecah(kata)
    for pasangan in merges:
        vocab_sementara = gabung({simbol: 1}, pasangan)
        simbol = next(iter(vocab_sementara))
    return list(simbol)

for k in ["makanan", "diminum", "meminum", "dimakannya"]:
    print(f"{k:<12} → {encode_bpe(k, merges)}")

## 3. Melatih Tokenizer dengan Library `tokenizers`

Implementasi di atas lambat untuk korpus besar. Library `tokenizers` (Rust, sangat cepat) melakukan hal yang sama. Kita latih dua jenis tokenizer di 500 kalimat NusaX dengan vocab 2.000:

- **BPE**: dipakai GPT-2, GPT-4, LLaMA. Memilih merge berdasarkan **frekuensi pasangan**.
- **WordPiece**: dipakai BERT. Mirip BPE, tetapi memilih merge berdasarkan **likelihood** (seberapa "berguna" gabungan itu), dan menandai potongan lanjutan kata dengan awalan `##`.

Jenis ketiga, **Unigram** (SentencePiece, dipakai T5 dan XLM-R), bekerja terbalik: mulai dari vocab besar lalu memangkasnya. Unigram menjadi bahan Latihan.

In [ ]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, decoders

def latih_tokenizer(jenis, vocab_size=2000):
    if jenis == "bpe":
        tok = Tokenizer(models.BPE(unk_token="[UNK]", end_of_word_suffix="</w>"))
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]", "[PAD]"],
                                      end_of_word_suffix="</w>")
        tok.decoder = decoders.BPEDecoder(suffix="</w>")
    else:
        tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
        trainer = trainers.WordPieceTrainer(vocab_size=vocab_size,
                                            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"])
        tok.decoder = decoders.WordPiece()
    tok.normalizer = normalizers.Sequence([normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents()])
    tok.pre_tokenizer = pre_tokenizers.Whitespace()     # pisahkan per kata & tanda baca dulu
    tok.train_from_iterator(korpus, trainer)
    return tok

tok_bpe = latih_tokenizer("bpe")
tok_wp = latih_tokenizer("wordpiece")

kalimat = "Pelayanannya sangat mengecewakan, makanannya juga kemahalan!"
for nama, tok in [("BPE", tok_bpe), ("WordPiece", tok_wp)]:
    enc = tok.encode(kalimat)
    print(f"{nama:<10} ({len(enc.tokens)} token): {enc.tokens}")
    print(f"{'':<10} ids: {enc.ids}")
    print(f"{'':<10} decode: {tok.decode(enc.ids)!r}")

Tokenizer yang dilatih di data sendiri memecah kata umum dalam korpus (`makanan`, `pelayanan`) menjadi sedikit token, dan memecah kata jarang menjadi potongan-potongan kecil.

## 4. Tokenizer Pretrained pada Bahasa Indonesia

Setiap model pretrained membawa tokenizer-nya sendiri, dan **keduanya harus selalu dipakai berpasangan**. Kita bandingkan empat tokenizer dari Hugging Face Hub:

| Model | Algoritma | Dilatih pada |
|---|---|---|
| `gpt2` | BPE tingkat byte | teks bahasa Inggris |
| `bert-base-uncased` | WordPiece | teks bahasa Inggris |
| `indobenchmark/indobert-base-p1` | WordPiece | teks bahasa Indonesia (Indo4B) |
| `xlm-roberta-base` | SentencePiece (Unigram) | 100 bahasa |

Ukuran yang dipakai: **fertility** = rata-rata jumlah token per kata. Semakin kecil, semakin efisien tokenizer untuk bahasa tersebut: input lebih pendek, komputasi lebih murah, dan (untuk API LLM) biaya lebih rendah.

In [ ]:
from transformers import AutoTokenizer

NAMA_MODEL = ["gpt2", "bert-base-uncased", "indobenchmark/indobert-base-p1", "xlm-roberta-base"]
tokenizers_hub = {n: AutoTokenizer.from_pretrained(n) for n in NAMA_MODEL}

contoh = "Pelayanannya sangat mengecewakan, makanannya juga kemahalan!"
for nama, tok in tokenizers_hub.items():
    t = tok.tokenize(contoh)
    print(f"{nama:<32} ({len(t):2d} token) {t}")

Hitung fertility di seluruh kalimat test NusaX (bahasa Indonesia) untuk keempat tokenizer pretrained dan dua tokenizer buatan kita:

In [ ]:
def fertility(tokenize_fn, teks_list):
    n_token = sum(len(tokenize_fn(t)) for t in teks_list)
    n_kata = sum(len(t.split()) for t in teks_list)
    return n_token / n_kata

teks_test = test_df["text"].tolist()
hasil = {nama: fertility(tok.tokenize, teks_test) for nama, tok in tokenizers_hub.items()}
hasil["BPE buatan sendiri (2k)"] = fertility(lambda t: tok_bpe.encode(t).tokens, teks_test)
hasil["WordPiece buatan sendiri (2k)"] = fertility(lambda t: tok_wp.encode(t).tokens, teks_test)

for nama, f in sorted(hasil.items(), key=lambda x: x[1]):
    print(f"{nama:<32} {f:.2f} token/kata  {'█' * int(f * 10)}")

plt.barh(list(hasil), list(hasil.values()))
plt.xlabel("token per kata (lebih kecil = lebih efisien)"); plt.title("Fertility pada teks bahasa Indonesia")
plt.gca().invert_yaxis(); plt.show()

Biasanya tokenizer yang dilatih pada bahasa Inggris (GPT-2, BERT) jauh lebih boros untuk bahasa Indonesia daripada IndoBERT. Konsekuensinya: teks yang sama memakan lebih banyak token dari **context window** model, dan biaya API LLM yang dihitung per token juga lebih mahal. Inilah salah satu alasan kita memakai **IndoBERT** di notebook 03.

## 5. Anatomi Output Tokenizer Hugging Face

Memanggil tokenizer langsung (bukan `.tokenize`) menghasilkan semua yang dibutuhkan model:
- **`input_ids`**: ID token, termasuk **token spesial** (`[CLS]` di awal dan `[SEP]` di akhir untuk BERT).
- **`attention_mask`**: `1` untuk token asli, `0` untuk **padding**, agar model mengabaikan padding.
- **Padding** menyamakan panjang dalam satu batch; **truncation** memotong teks yang melebihi `max_length`.

In [ ]:
tok = tokenizers_hub["indobenchmark/indobert-base-p1"]
batch = tok(["Enak banget!", "Pelayanannya lambat dan pegawainya tidak ramah."],
            padding=True, truncation=True, max_length=12, return_tensors="pt")

for k, v in batch.items():
    print(f"{k:<15} {tuple(v.shape)}\n{v}\n")
for i in range(2):
    print(f"kalimat {i}:", tok.convert_ids_to_tokens(batch["input_ids"][i]))
print("\ndecode:", tok.decode(batch["input_ids"][1], skip_special_tokens=True))
print("token spesial:", tok.special_tokens_map)

## Ringkasan

- Tokenisasi **subword** (BPE, WordPiece, Unigram) menyeimbangkan ukuran vocab dan panjang deretan, serta menghilangkan masalah kata tidak dikenal (OOV).
- **BPE**: mulai dari karakter, gabungkan pasangan tersering secara berulang; aturan merge diterapkan berurutan ke teks baru.
- Model dan tokenizer **selalu berpasangan**; jangan mencampur tokenizer model lain.
- Tokenizer yang dilatih pada bahasa Inggris **boros token** untuk bahasa Indonesia; pilih model yang tokenizer-nya cocok dengan bahasa data.
- Output tokenizer HF: `input_ids`, `attention_mask`, token spesial, padding, dan truncation.

## Latihan

1. **Ukuran vocab.** Latih ulang tokenizer BPE di korpus NusaX dengan `vocab_size` 500, 1.000, 4.000, dan 8.000. Plot fertility di data test terhadap ukuran vocab. Di titik mana penambahan vocab tidak lagi banyak membantu, dan apa biaya dari vocab yang terlalu besar (petunjuk: ukuran layer embedding dan output model)?
2. **Unigram.** Latih tokenizer **Unigram** (`models.Unigram()` + `trainers.UnigramTrainer`) dengan vocab 2.000 dan bandingkan segmentasinya dengan BPE dan WordPiece pada 5 kalimat berimbuhan (misalnya `"ketidakberdayaannya"`, `"dipertanggungjawabkan"`). Tokenizer mana yang potongannya paling mendekati morfem (akar kata + imbuhan) bahasa Indonesia?
3. **Biaya token.** Ambil 3 paragraf berita berbahasa Indonesia dan terjemahan bahasa Inggrisnya. Hitung jumlah token keduanya dengan tokenizer `gpt2` dan `xlm-roberta-base`. Berapa persen lebih mahal memproses versi Indonesia jika biaya API dihitung per token? Apa implikasinya saat merancang aplikasi LLM untuk pengguna Indonesia?